In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import warnings

warnings.filterwarnings("ignore")

In [2]:
df_mart = pd.read_pickle('df_mart.pkl')
df_mart = df_mart.rename(columns={'delay_status' : 'is_delay'})

print(df_mart.info())
display(df_mart)

<class 'pandas.DataFrame'>
RangeIndex: 87027 entries, 0 to 87026
Data columns (total 9 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       87027 non-null  str           
 1   order_delivered_customer_date  87027 non-null  datetime64[us]
 2   order_estimated_delivery_date  87027 non-null  datetime64[us]
 3   review_score                   87027 non-null  float64       
 4   customer_state                 87027 non-null  str           
 5   diff_delivery_date             87027 non-null  float64       
 6   is_delay                       87027 non-null  int64         
 7   is_extreme_delay               87027 non-null  int64         
 8   is_low_score                   87027 non-null  int64         
dtypes: datetime64[us](2), float64(2), int64(3), str(2)
memory usage: 6.0 MB
None


,order_id,order_delivered_customer_date,order_estimated_delivery_date,review_score,customer_state,diff_delivery_date,is_delay,is_extreme_delay,is_low_score
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-10 21:25:13,2017-10-18,4.0,SP,7.11,0,0,0
1,53cdb2fc8bc7dce0b6741e2150273451,2018-08-07 15:27:45,2018-08-13,4.0,BA,5.36,0,0,0
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-17 18:06:29,2018-09-04,5.0,GO,17.25,0,0,0
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-12-02 00:28:42,2017-12-15,5.0,RN,12.98,0,0,0
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-16 18:17:02,2018-02-26,5.0,SP,9.24,0,0,0
...,...,...,...,...,...,...,...,...,...
87022,9c5dedf39a927c1b2549525ed64a053c,2017-03-17 15:08:01,2017-03-28,5.0,SP,10.37,0,0,0
87023,63943bddc261676b46f01ca7ac2f7bd8,2018-02-28 17:37:56,2018-03-02,4.0,SP,1.27,0,0,0
87024,83c1379a015df1e13d02aae0204711ab,2017-09-21 11:24:17,2017-09-27,5.0,BA,5.52,0,0,0
87025,11c177c8e97725db2631073c19f07b62,2018-01-25 23:32:54,2018-02-15,2.0,RJ,20.02,0,0,1


In [3]:
df_state = df_mart[df_mart['is_delay'] == 1].groupby('customer_state').agg(
    total_delayed_order = ('order_id', 'count'),
    low_score_rate_delay = ('is_low_score', 'mean'),
    extreme_delayed_order = ('is_extreme_delay', 'sum'),      # count asli, integer
    extreme_delay_rate = ('is_extreme_delay', 'mean')          # rate asli, presisi penuh
).sort_values(by='total_delayed_order', ascending=False).reset_index()

# baru bulatkan untuk tampilan, SETELAH semua perhitungan selesai
df_state['low_score_rate_delay'] = (df_state['low_score_rate_delay'] * 100).round(2)
df_state['extreme_delay_rate'] = (df_state['extreme_delay_rate'] * 100).round(2)

# Minimum Sample Threshold --> Central Limit Theorem (n = 30)
min_sample = 30

# State Ranking
state_ranking = df_state[df_state['total_delayed_order'] >= min_sample].copy()

# Priority Threshold
volume_th = state_ranking['total_delayed_order'].quantile(0.75)
impact_th = state_ranking['low_score_rate_delay'].quantile(0.75)

state_ranking['flag_volume'] = state_ranking['total_delayed_order'] >= volume_th
state_ranking['flag_impact'] = state_ranking['low_score_rate_delay'] >= impact_th

state_ranking['priority_status'] = 'Low Priority'
state_ranking.loc[state_ranking['flag_volume'] | state_ranking['flag_impact'], 'priority_status'] = 'Medium Priority'
state_ranking.loc[state_ranking['flag_volume'] & state_ranking['flag_impact'], 'priority_status'] = 'High Priority'

state_ranking['ranking_number'] = np.where(
    state_ranking['priority_status'] == 'High Priority',
    1, 
    np.where(
        state_ranking['priority_status'] == 'Medium Priority',
        2,
        3
    )
)

# Final Priority
final_priority = state_ranking.sort_values(
    by='ranking_number', ascending=True
).reset_index(drop=True).drop(columns='ranking_number')

display(final_priority)

,customer_state,total_delayed_order,low_score_rate_delay,extreme_delayed_order,extreme_delay_rate,flag_volume,flag_impact,priority_status
0,RJ,1273,66.54,145,11.39,True,True,High Priority
1,SP,1445,41.45,57,3.94,True,False,Medium Priority
2,MG,426,46.95,14,3.29,True,False,Medium Priority
3,BA,374,52.67,25,6.68,True,False,Medium Priority
4,RS,293,54.61,12,4.10,True,False,Medium Priority
5,SC,256,49.22,3,1.17,True,False,Medium Priority
6,PA,93,64.52,5,5.38,False,True,Medium Priority
7,PE,137,67.88,11,8.03,False,True,Medium Priority
8,AL,77,67.53,4,5.19,False,True,Medium Priority
9,SE,40,67.50,3,7.50,False,True,Medium Priority


In [4]:
# print(final_priority['flag_volume'].corr(final_priority['flag_impact']))
# print(final_priority['total_delayed_order'].corr(final_priority['low_score_rate_delay']))

print(volume_th)
print(impact_th)

print(df_mart.info())
print(final_priority.info())

256.0
63.16
<class 'pandas.DataFrame'>
RangeIndex: 87027 entries, 0 to 87026
Data columns (total 9 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       87027 non-null  str           
 1   order_delivered_customer_date  87027 non-null  datetime64[us]
 2   order_estimated_delivery_date  87027 non-null  datetime64[us]
 3   review_score                   87027 non-null  float64       
 4   customer_state                 87027 non-null  str           
 5   diff_delivery_date             87027 non-null  float64       
 6   is_delay                       87027 non-null  int64         
 7   is_extreme_delay               87027 non-null  int64         
 8   is_low_score                   87027 non-null  int64         
dtypes: datetime64[us](2), float64(2), int64(3), str(2)
memory usage: 6.0 MB
None
<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns 

In [5]:
display(df_mart)
display(df_state)
display(final_priority)

,order_id,order_delivered_customer_date,order_estimated_delivery_date,review_score,customer_state,diff_delivery_date,is_delay,is_extreme_delay,is_low_score
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-10 21:25:13,2017-10-18,4.0,SP,7.11,0,0,0
1,53cdb2fc8bc7dce0b6741e2150273451,2018-08-07 15:27:45,2018-08-13,4.0,BA,5.36,0,0,0
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-17 18:06:29,2018-09-04,5.0,GO,17.25,0,0,0
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-12-02 00:28:42,2017-12-15,5.0,RN,12.98,0,0,0
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-16 18:17:02,2018-02-26,5.0,SP,9.24,0,0,0
...,...,...,...,...,...,...,...,...,...
87022,9c5dedf39a927c1b2549525ed64a053c,2017-03-17 15:08:01,2017-03-28,5.0,SP,10.37,0,0,0
87023,63943bddc261676b46f01ca7ac2f7bd8,2018-02-28 17:37:56,2018-03-02,4.0,SP,1.27,0,0,0
87024,83c1379a015df1e13d02aae0204711ab,2017-09-21 11:24:17,2017-09-27,5.0,BA,5.52,0,0,0
87025,11c177c8e97725db2631073c19f07b62,2018-01-25 23:32:54,2018-02-15,2.0,RJ,20.02,0,0,1


,customer_state,total_delayed_order,low_score_rate_delay,extreme_delayed_order,extreme_delay_rate
0,SP,1445,41.45,57,3.94
1,RJ,1273,66.54,145,11.39
2,MG,426,46.95,14,3.29
3,BA,374,52.67,25,6.68
4,RS,293,54.61,12,4.10
5,SC,256,49.22,3,1.17
6,ES,189,46.56,8,4.23
7,PR,177,44.07,8,4.52
8,CE,165,63.03,22,13.33
9,PE,137,67.88,11,8.03


,customer_state,total_delayed_order,low_score_rate_delay,extreme_delayed_order,extreme_delay_rate,flag_volume,flag_impact,priority_status
0,RJ,1273,66.54,145,11.39,True,True,High Priority
1,SP,1445,41.45,57,3.94,True,False,Medium Priority
2,MG,426,46.95,14,3.29,True,False,Medium Priority
3,BA,374,52.67,25,6.68,True,False,Medium Priority
4,RS,293,54.61,12,4.10,True,False,Medium Priority
5,SC,256,49.22,3,1.17,True,False,Medium Priority
6,PA,93,64.52,5,5.38,False,True,Medium Priority
7,PE,137,67.88,11,8.03,False,True,Medium Priority
8,AL,77,67.53,4,5.19,False,True,Medium Priority
9,SE,40,67.50,3,7.50,False,True,Medium Priority


In [6]:
# Ambil SEMUA wilayah yang punya pesanan delay (termasuk yang <30)
all_state_delay = df_mart[df_mart['is_delay'] == 1].groupby('customer_state').agg(
    total_delayed_order=('order_id', 'count'),
    low_score_rate_delay=('is_low_score', 'mean'),
    extreme_delayed_order=('is_extreme_delay', 'sum'),
    extreme_delay_rate=('is_extreme_delay', 'mean')
).reset_index()

all_state_delay['low_score_rate_delay'] = (all_state_delay['low_score_rate_delay'] * 100).round(2)
all_state_delay['extreme_delay_rate'] = (all_state_delay['extreme_delay_rate'] * 100).round(2)

# Cari wilayah yang BELUM ada di final_priority (yaitu yang <30 delayed order)
not_classified_states = all_state_delay[
    ~all_state_delay['customer_state'].isin(final_priority['customer_state'])
].copy()

# Beri flag dan status yang sesuai
not_classified_states['flag_volume'] = False
not_classified_states['flag_impact'] = False
not_classified_states['priority_status'] = 'Not Classified'

# Gabungkan ke final_priority
final_priority_full = pd.concat([final_priority, not_classified_states], ignore_index=True)

# Cek hasilnya - harus 27 baris total
print(final_priority_full.shape)
print(final_priority_full['priority_status'].value_counts())

(27, 8)
priority_status
Medium Priority    10
Low Priority       10
Not Classified      6
High Priority       1
Name: count, dtype: int64


In [7]:
display(final_priority_full)

,customer_state,total_delayed_order,low_score_rate_delay,extreme_delayed_order,extreme_delay_rate,flag_volume,flag_impact,priority_status
0,RJ,1273,66.54,145,11.39,True,True,High Priority
1,SP,1445,41.45,57,3.94,True,False,Medium Priority
2,MG,426,46.95,14,3.29,True,False,Medium Priority
3,BA,374,52.67,25,6.68,True,False,Medium Priority
4,RS,293,54.61,12,4.10,True,False,Medium Priority
5,SC,256,49.22,3,1.17,True,False,Medium Priority
6,PA,93,64.52,5,5.38,False,True,Medium Priority
7,PE,137,67.88,11,8.03,False,True,Medium Priority
8,AL,77,67.53,4,5.19,False,True,Medium Priority
9,SE,40,67.50,3,7.50,False,True,Medium Priority


In [8]:
# df_mart.to_csv('olist_data_mart.csv', index=False, encoding='utf-8-sig')
# final_priority_full.to_csv('olist_state_priority.csv', index=False, encoding='utf-8-sig')

In [14]:
print(df_mart[df_mart['order_id'] == '8563039e855156e48fccee4d611a3196']['diff_delivery_date'])

30   -0.04
Name: diff_delivery_date, dtype: float64
